In [ ]:
### think about units
### how to apply to events?

In [1]:
# import xarray as xr
# import matplotlib.pyplot as plt
# nc_path ="../../Data/SoilDynamicVariables/combined_from_tif/Ens_01/nc/23/soil_23_2_1994_7_11_Ens_01_vol_5km.nc"
# hc_5km = xr.open_dataset(nc_path, engine="netcdf4")

# import os

# print(nc_path)
# print(os.path.exists(nc_path))
# print(os.path.getsize(nc_path))
# plt.pcolormesh(hc_5km['remaining_capacity'])

# hc_5km['remaining_capacity'].data


In [39]:
import glob
import numpy as np
import xarray as xr

# ------------------------------------------------------------
# Hydraulic conductivity → effective soil depth
# ------------------------------------------------------------

hc_5km = xr.open_dataset("../../Data/HydraulicConductivity/GB_5km.nc")

# Extract hydraulic conductivity DataArray
Ks = hc_5km["aggregated_mean"]

# Calculate effective soil depth
L_5km = 4 * np.sqrt(Ks)

# Rename variable
L_5km = L_5km.rename("effective_soil_depth")

# Set units
L_5km.attrs["units"] = "m"

# Convert to mm
L_5km_mm = L_5km * 1000


# ------------------------------------------------------------
# Effective porosity
# ------------------------------------------------------------

ep_5km = xr.open_dataset( "../../Data/EffectivePorosity/GB_5km_mean.nc")["aggregated_mean"]


# # ------------------------------------------------------------
# # Load soil moisture
# # ------------------------------------------------------------

# ens = "01"
# year = 1990

# sm_dir = (f"/scratch/hydro4/shared_data/climate_projections/UKCP18/UKCP_local/Soil_moisture/5km_regridded/"f"Ens_{ens}/")

# sm_file = glob.glob(f"{sm_dir}/r001i1p*****_{year-1}1201-{year}1130_mrso.nc")[0]

# sm_ds = xr.open_dataset(sm_file)

# mrso = sm_ds["moisture_content_of_soil_layer"]

# # mrso is kg/m², which is equivalent to mm of water
# water_depth_mm = mrso


# # ------------------------------------------------------------
# # Volumetric water content
# # ------------------------------------------------------------

# vol_water_content = water_depth_mm / L_5km_mm


# # ------------------------------------------------------------
# # Degree of saturation
# # ------------------------------------------------------------

# saturation = vol_water_content / ep_5km


# # ------------------------------------------------------------
# # Physical limits
# # ------------------------------------------------------------

# saturation = saturation.where(saturation >= 0)
# saturation = saturation.clip(max=1)


# # ------------------------------------------------------------
# # Maximum infiltration volume
# # ------------------------------------------------------------

# max_infiltration_vol = ep_5km * L_5km_mm


# # ------------------------------------------------------------
# # Remaining infiltration volume
# # ------------------------------------------------------------

# remain_infiltration_vol = saturation * max_infiltration_vol

In [9]:
import iris
import iris.cube
import numpy as np
import glob
from pathlib import Path
from tqdm import tqdm
import xarray as xr

# output_dir = Path("../../Data/Soil_saturation/saturation_5km/")

# output_dir.mkdir(parents=True, exist_ok=True)

# soil_depth_mm = L_5km.copy()
# soil_depth_mm.data = L_5km.data * 1000

In [40]:
# effective_porosity = iris.load("../../Data/EffectivePorosity/GB_5km_mean.nc")[0]
# effective_porosity = xr.open_dataset("../../Data/EffectivePorosity/GB_5km_mean.nc", engine="netcdf4")
sm_cube = xr.open_dataset(sm_file)
soil_depth_mm = 225

In [18]:
ens='01'
sm_dir = (f"/scratch/hydro4/shared_data/climate_projections/UKCP18/UKCP_local/Soil_moisture/5km_regridded/Ens_{ens}/")
year=1990
soil_depth_mm = 225

output_file = output_dir / f"soil_saturation_Ens_{ens}_{year}.nc"

# Skip if already processed
# if output_file.exists():
#     continue

sm_file = glob.glob(f"{sm_dir}/r001i1p*****_{year-1}1201-{year}1130_mrso.nc")[0]


# --------------------------------------------------
# Load one year
# --------------------------------------------------

sm_cube = xr.open_dataset(sm_file)

# --------------------------------------------------
# Convert masked -> NaN
# --------------------------------------------------

if np.ma.is_masked(sm_cube['moisture_content_of_soil_layer']):
    sm_data = sm_cube['moisture_content_of_soil_layer'].filled(np.nan)
else:
    sm_data = sm_cube['moisture_content_of_soil_layer']

# --------------------------------------------------
# Water depth
# kg/m2 == mm
# --------------------------------------------------

water_depth_mm = sm_data

# --------------------------------------------------
# Volumetric water content
# --------------------------------------------------

theta = (water_depth_mm /soil_depth_mm)

# --------------------------------------------------
# Saturation
# --------------------------------------------------

saturation = (theta /effective_porosity['aggregated_mean'])

# --------------------------------------------------
# Keep physically sensible values
# --------------------------------------------------

#         saturation = np.where(saturation < 0,np.nan,saturation)
#         saturation = np.where(saturation > 1,1,saturation)

saturation = saturation.where(saturation >= 0)
saturation = saturation.clip(max=1)

# --------------------------------------------------
# Max storage depth
# --------------------------------------------------
max_storage_depth = effective_porosity['aggregated_mean'] * L_5km_mm

remaining_storage_depth = ((1 - saturation) * max_storage_depth)


# --------------------------------------------------
# Put result into cube
# --------------------------------------------------

sat_cube = sm_cube.copy(data=saturation.astype(np.float32))

sat_cube.rename("degree_of_saturation")
sat_cube.units = "1"

# --------------------------------------------------
# Save
# --------------------------------------------------
# print(output_file)
# iris.save(sat_cube,str(output_file))


NameError: name 'effective_porosity' is not defined

In [38]:
for ens in ['01', '04']:

    print(f"\nProcessing ensemble {ens}")

    sm_dir = (f"/scratch/hydro4/shared_data/climate_projections/UKCP18/UKCP_local/Soil_moisture/5km_regridded/Ens_{ens}/")

    for year in tqdm(range(1980, 1982), desc=f"Ens {ens}"):

        output_file = output_dir / f"soil_saturation_Ens_{ens}_{year}.nc"

        # Skip if already processed
        if output_file.exists():
            continue

        try:
            sm_file = glob.glob(f"{sm_dir}/r001i1p*****_{year-1}1201-{year}1130_mrso.nc")[0]
        except IndexError:
            continue

        # --------------------------------------------------
        # Load one year
        # --------------------------------------------------

        sm_cube = xr.open_dataset(sm_file)

        # --------------------------------------------------
        # Convert masked -> NaN
        # --------------------------------------------------

        if np.ma.is_masked(sm_cube['moisture_content_of_soil_layer']):
            sm_data = sm_cube['moisture_content_of_soil_layer'].filled(np.nan)
        else:
            sm_data = sm_cube['moisture_content_of_soil_layer']

        # --------------------------------------------------
        # Water depth
        # kg/m2 == mm
        # --------------------------------------------------

        water_depth_mm = sm_data

        # --------------------------------------------------
        # Volumetric water content
        # --------------------------------------------------

        theta = (water_depth_mm /soil_depth_mm.data)
        
        # --------------------------------------------------
        # Saturation
        # --------------------------------------------------

        saturation = (theta /effective_porosity['aggregated_mean'])

        # --------------------------------------------------
        # Keep physically sensible values
        # --------------------------------------------------

#         saturation = np.where(saturation < 0,np.nan,saturation)
#         saturation = np.where(saturation > 1,1,saturation)

        saturation = saturation.where(saturation >= 0)
        saturation = saturation.clip(max=1)
        
        # --------------------------------------------------
        # Max storage depth
        # --------------------------------------------------
        max_storage_depth = effective_porosity['aggregated_mean'] * L_5km_mm

        remaining_storage_depth = ((1 - saturation) * max_storage_depth)
        
        
        # --------------------------------------------------
        # Put result into cube
        # --------------------------------------------------

        sat_cube = sm_cube.copy(data=saturation.astype(np.float32))

        sat_cube.rename("degree_of_saturation")
        sat_cube.units = "1"

        # --------------------------------------------------
        # Save
        # --------------------------------------------------
        print(output_file)
        iris.save(sat_cube,str(output_file))

#         # Explicitly release memory
#         del sm_cube
#         del sm_data
#         del water_depth_mm
#         del theta
#         del saturation
#         del sat_cube


Processing ensemble 01


Ens 01:  50%|███████████████████████████████████████████████████████████                                                           | 1/2 [00:02<00:02,  2.87s/it]


AttributeError: 'function' object has no attribute 'rename'